# Supervised Learning

## Introduction {#sec-08-top}
## Classification versus Regression
## Supervised Learning Workflow
## Scikit-learn
### Input Data Structure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from itables import show

from lime import lime_tabular

from sklearn import tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
  accuracy_score, f1_score, confusion_matrix, 
  ConfusionMatrixDisplay, precision_score, recall_score, r2_score
  )
from sklearn.model_selection import (
    cross_val_score, cross_validate, ShuffleSplit, 
    learning_curve, validation_curve, GridSearchCV, train_test_split
    )
from sklearn.inspection import permutation_importance
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import PartialDependenceDisplay, partial_dependence

## Measures of Performance
### For Classification
#### Accuracy 
#### Precision and Recall
#### F1 score
### For Regression
#### Root Mean Squared Error
#### Mean Absolute Error
## Classification
### Example: Heart failure

In [ ]:
hf = pd.read_csv("data/heart+failure+clinical+records/"+
                 "heart_failure_clinical_records_dataset.csv")
print(hf.head())

### Decision Tree {#sec-08-dec-tree}

In [ ]:
y = hf.DEATH_EVENT
X = hf.iloc[:, 0:12]

clf = tree.DecisionTreeClassifier(max_depth=4)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.25, 
                                                 random_state=41, stratify=y)

In [ ]:
print(f"The proportion of 1's in the overall data is {y.mean():.3f}.")
print(f"The proportion of 1's in the training data is {y_train.mean():.3f}.")
print(f"The proportion of 1's in the test data is {y_test.mean():.3f}.")

### Example: Heart failure decision tree

In [ ]:
clf.fit(X_train, y_train,);

In [ ]:
clf.predict_proba(X_train.sample(random_state=3005))

In [ ]:
#| fig-align: center
#| fig-cap: "Heart failure decision tree"
#| label: fig-heart-dt
#| fig-pos: 'ht'

plt.figure(figsize =(18, 6))
tree.plot_tree(clf,feature_names=X.columns, filled=True, max_depth=2);

### Example: Heart failure classification scores

In [ ]:
#| fig-align: center
#| label: fig-confusion-training
#| fig-cap: "Confusion matrix, training set"
#| fig-pos: 'ht'

y_pred_train = clf.predict(X_train)
ConfusionMatrixDisplay.from_predictions(y_train, y_pred_train, 
                                        text_kw={'size': 'xx-large'},
                                        labels=clf.classes_, cmap='bone');
print(f"""
##: For training set:
----
The precision (for cat. 1) is {precision_score(y_train, y_pred_train):.3f}
The recall (for cat. 1) is {recall_score(y_train, y_pred_train):.3f}
The accuracy (for cat. 1) is {accuracy_score(y_train, y_pred_train):.3f}
The f1-score (for cat. 1) is {f1_score(y_train, y_pred_train):.3f}
""")

In [ ]:
#| fig-align: center
#| label: fig-confusion-test
#| fig-cap: "Confusion matrix, test set"
#| fig-pos: 'ht'

y_pred_test = clf.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test,  
                                        text_kw={'size': 'xx-large'},
                                        labels=clf.classes_, cmap='bone');
print(f"""
##: For test set:
----
The accuracy (for cat. 1) is {accuracy_score(y_test, y_pred_test):.3f}
The precision (for cat. 1) is {precision_score(y_test, y_pred_test):.3f}
The recall (for cat. 1) is {recall_score(y_test, y_pred_test):.3f}
The f1-score (for cat. 1) is {f1_score(y_test, y_pred_test):.3f}
""")

### Variable importance
### Example: Heart failure permutation based

In [ ]:
#| fig-align: center
#| label: fig-var-impt-perm
#| fig-cap: "Variable importance, permutation approach"
#| fig-pos: 'ht'

result = permutation_importance(clf, X_test, y_test,  
                                n_repeats=30, random_state=42)

sorted_importances_idx = result.importances_mean.argsort()
importances = pd.DataFrame(
    result.importances[sorted_importances_idx].T,
    columns=X.columns[sorted_importances_idx],
)
ax = importances.plot.box(vert=False, whis=10)
ax.set_title("Permutation Importances (test set)")
ax.axvline(x=0, color="k", linestyle="--")
ax.set_xlabel("Decrease in accuracy score")
ax.figure.tight_layout()

#### 
### Example: Heart failure partial dependence 

In [ ]:
X_train2 = X_train.copy()

X_train2['creatinine_phosphokinase'] = X_train2['creatinine_phosphokinase'].astype(float)
X_train2['serum_sodium'] = X_train2['serum_sodium'].astype(float)

In [ ]:
#| fig-align: center
#| label: fig-var-impt-pdp
#| fig-cap: "Variable importance, partial dependence"
#| fig-pos: 'ht'
 
_, ax = plt.subplots(ncols=3, nrows=2, figsize=(12, 6), constrained_layout=True)
features_info = {
    "features": ["age", "creatinine_phosphokinase", "platelets", 
                 "serum_creatinine", "serum_sodium", ("age", "serum_creatinine"),
                ],
    "kind": "average",
}
display = PartialDependenceDisplay.from_estimator(
    clf,
    X_train2,
    **features_info,
    ax=ax,
    contour_kw = {'cmap': "Reds"}
)

### Random Forest {#sec-08-forest}
### Example: Heart failure grid search

In [ ]:
p_range = range(1, 11, 1)
#list(p_range)
cv_search = GridSearchCV(RandomForestClassifier(n_estimators=20, random_state=21), 
                         return_train_score=True,
                         param_grid ={'max_depth': p_range},
                         scoring = 'accuracy', cv= 5, verbose=1)
cv_search.fit(X_train, y_train);

In [ ]:
cv_search.best_estimator_;

In [ ]:
#| fig-align: center
#| label: fig-valid-curve
#| fig-cap: "Validation curve, random forest"

train_means = cv_search.cv_results_['mean_train_score']
train_sd = cv_search.cv_results_['std_train_score']

test_means = cv_search.cv_results_['mean_test_score']
test_sd = cv_search.cv_results_['std_test_score']
plt.plot(p_range, train_means, 'o-', label='Training', color='blue')
plt.fill_between(p_range, train_means-train_sd, train_means+train_sd, 
                 color='blue', alpha=0.2)

plt.plot(p_range, test_means, 'o-', label='CV (Test)', color='red')
plt.fill_between(p_range, test_means-test_sd, test_means+test_sd, 
                 color='red', alpha=0.2)

plt.legend(loc='lower right');plt.ylabel('Accuracy');
plt.xlabel('Complexity');plt.title('Validation Curve');

### Example: Heart failure random forest classification scores

In [ ]:
rf = RandomForestClassifier(n_estimators=20, max_depth=3, random_state=40)
rf.fit(X_train, y_train);

In [ ]:
y_pred_train = rf.predict(X_train)
print(f"""
For training set:
-----------------
The precision (for cat. 1) is {precision_score(y_train, y_pred_train):.3f}
The recall (for cat. 1) is {recall_score(y_train, y_pred_train):.3f}
The accuracy (for cat. 1) is {accuracy_score(y_train, y_pred_train):.3f}
The f1-score (for cat. 1) is {f1_score(y_train, y_pred_train):.3f}
""")

In [ ]:
y_pred_test = rf.predict(X_test)
print(f"""
For test set:
-------------
The precision (for cat. 1) is {precision_score(y_test, y_pred_test):.3f}
The recall (for cat. 1) is {recall_score(y_test, y_pred_test):.3f}
The accuracy (for cat. 1) is {accuracy_score(y_test, y_pred_test):.3f}
The f1-score (for cat. 1) is {f1_score(y_test, y_pred_test):.3f}
""")

## Regression

In [ ]:
re2 = pd.read_csv("data/taiwan_dataset.csv")

X_re = re2.loc[:, ['trans_date', 'house_age', 'dist_MRT', 
                   'num_stores', 'Xs', 'Ys']]
re_scaler = StandardScaler().fit(X_re)
X_re_scaled = re_scaler.transform(X_re)
y_re = re2.price

Xre_train,Xre_test, yre_train,yre_test = train_test_split(X_re_scaled, 
                                                          y_re, test_size=0.2, 
                                                          random_state=41)

### Random Forest Regressor

In [ ]:
p_range = range(1, 11, 1)
rf_search = GridSearchCV(RandomForestRegressor(n_estimators=10, random_state=43), 
                         {'max_depth': p_range}, 
                         scoring='r2', cv=5, verbose=1, return_train_score=True)
rf_search.fit(Xre_train, yre_train,)
rf_search.best_estimator_;

In [ ]:
#| fig-align: center
#| label: fig-taiwan-random-forest
#| fig-cap: "Validation curve, random forest for Taiwan data"
train_means = rf_search.cv_results_['mean_train_score']
train_sd = rf_search.cv_results_['std_train_score']

test_means = rf_search.cv_results_['mean_test_score']
test_sd = rf_search.cv_results_['std_test_score']

plt.plot(p_range, train_means, 'o-', label='Training', color='blue')
plt.fill_between(p_range, train_means-train_sd, train_means+train_sd, 
                 color='blue', alpha=0.2)

plt.plot(p_range, test_means, 'o-', label='CV (Test)', color='red')
plt.fill_between(p_range, test_means-test_sd, test_means+test_sd, 
                 color='red', alpha=0.2)

plt.legend(loc='lower right');plt.ylabel('R2');
plt.xlabel('Complexity');plt.title('Validation Curve');

### Example: Taiwan data random forest regression

In [ ]:
rf1 = RandomForestRegressor(n_estimators=10, max_depth = 2, random_state=89)
rf1.fit(Xre_train, yre_train)
yrf_pred = rf1.predict(Xre_test)
r2_score(yre_test, yrf_pred)

## Interpretability of Models {#sec-08-interpretability}
### LIME 
### Example: Taiwan data LIME

In [ ]:
re_scaler.inverse_transform(Xre_test[4, :].reshape(1, -1)).round()
#X_re.mean(axis=0).round(3)
#Xre_test[4, :].round(4)

In [ ]:
explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(Xre_train),
    feature_names=X_re.columns,
    mode='regression',
)
exp = explainer.explain_instance(
    data_row=Xre_test[4, :], 
    predict_fn=rf1.predict,
)

In [ ]:
#| fig-align: center
#| label: fig-taiwan-lime
#| fig-cap: "LIME Explanation for Taiwan data instance"
exp.as_pyplot_figure();

In [ ]:
#| include: false
# exp.show_in_notebook(show_table=True)

In [ ]:
print(f"The value predicted by the local approximation was: {exp.local_pred[0]:.3f}")

In [ ]:
s = 0.0
for x,y in exp.local_exp[1]:
    s += y

s + exp.intercept[0]

### ICE plots

In [ ]:
#| fig-align: center
_, ax = plt.subplots(ncols=2, nrows=1, figsize=(12, 4), constrained_layout=True)
features_info = {
    "features": [1,2], # no names in the array; 1 and 2 correspond to house_age and dist_MRT
    "kind": "both",
}
display = PartialDependenceDisplay.from_estimator(
    rf1,
    Xre_train,
    **features_info,
    ax=ax
)

## Summary
## References
### Website and video references
### Documentation references
## Exercises